# Milestone 1 — NLP Foundation & Semantic Similarity

This notebook covers the NLP foundation stage of the Smart MCQ Solver project.

Topics covered:
- Dataset loading and inspection
- Question-option text representation
- TF-IDF feature extraction
- Logistic Regression baseline
- Group-based validation
- MAP@3 evaluation

In [ ]:
import numpy as np
import pandas as pd

import kagglehub

In [ ]:
# ======================================================
# STEP 1 : IMPORTS & LOAD DATA
# ======================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

DATA_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge"

train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
test_df = pd.read_csv(f"{DATA_PATH}/test.csv")
sample_sub = pd.read_csv(f"{DATA_PATH}/sample_submission.csv")

print("="*60)
print("Train Shape :", train_df.shape)
print("Test Shape  :", test_df.shape)
print("="*60)

display(train_df.head())
display(test_df.head())
display(sample_sub.head())

In [ ]:
# ===============================================
# STEP 2 : DATASET OVERVIEW
# ===============================================

print("="*70)
print("Columns")
print("="*70)
print(train_df.columns.tolist())

print("\n")

print("="*70)
print("Dataset Info")
print("="*70)
train_df.info()

print("\n")

print("="*70)
print("Missing Values")
print("="*70)
print(train_df.isnull().sum())

print("\n")

print("="*70)
print("Answer Distribution")
print("="*70)
print(train_df["answer"].value_counts())

print("\n")

print("="*70)
print("Unique Answers")
print("="*70)
print(train_df["answer"].unique())

In [ ]:
# ======================================================
# STEP 3 : DATASET ANALYSIS
# ======================================================

print("=" * 70)
print("Duplicate Prompts")
print("=" * 70)
print(train_df["prompt"].duplicated().sum())

print("\n")

print("=" * 70)
print("Duplicate Entire Rows")
print("=" * 70)
print(train_df.duplicated().sum())

print("\n")

print("=" * 70)
print("Prompt Length Statistics")
print("=" * 70)

prompt_lengths = train_df["prompt"].str.split().str.len()

print(prompt_lengths.describe())

print("\n")

print("=" * 70)
print("Average Option Length")
print("=" * 70)

for option in ["A", "B", "C", "D", "E"]:
    lengths = train_df[option].str.split().str.len()
    print(f"{option}: Mean={lengths.mean():.2f}, Median={lengths.median():.0f}, Max={lengths.max()}")

In [ ]:
# ======================================================
# STEP 4 : ADVANCED EDA
# ======================================================

# Prompt lengths
train_df["prompt_len"] = train_df["prompt"].str.split().str.len()
test_df["prompt_len"] = test_df["prompt"].str.split().str.len()

print("="*70)
print("Prompt Length Comparison")
print("="*70)

print("Train")
print(train_df["prompt_len"].describe())

print("\nTest")
print(test_df["prompt_len"].describe())

# Option lengths
for option in ["A", "B", "C", "D", "E"]:
    train_df[f"{option}_len"] = train_df[option].str.split().str.len()

print("\n")
print("="*70)
print("Average Option Lengths")
print("="*70)

for option in ["A", "B", "C", "D", "E"]:
    print(f"{option}: {train_df[f'{option}_len'].mean():.2f}")

# Duplicate options within a question
duplicate_option_count = 0

for _, row in train_df.iterrows():
    opts = [row["A"], row["B"], row["C"], row["D"], row["E"]]
    if len(set(opts)) < 5:
        duplicate_option_count += 1

print("\n")
print("="*70)
print("Questions With Duplicate Options")
print("="*70)
print(duplicate_option_count)

# Answer distribution plot
plt.figure(figsize=(6,4))
train_df["answer"].value_counts().sort_index().plot(kind="bar")
plt.title("Answer Distribution")
plt.xlabel("Answer")
plt.ylabel("Count")
plt.show()

# Prompt length histogram
plt.figure(figsize=(8,4))
plt.hist(train_df["prompt_len"], bins=20)
plt.title("Prompt Length Distribution")
plt.xlabel("Words")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# ======================================================
# PHASE 2 - STEP 5 : MAP@3 METRIC
# ======================================================

def apk(actual, predicted, k=3):
    """
    Average Precision at K for one sample.
    """
    predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score


def mapk(actuals, predictions, k=3):
    """
    Mean Average Precision at K.
    """
    return np.mean([
        apk(a, p, k)
        for a, p in zip(actuals, predictions)
    ])


# ------------------------------------------------------

# Example

actual = ["A", "C", "D"]

predictions = [
    ["A", "B", "C"],
    ["B", "C", "A"],
    ["C", "A", "D"]
]

print("Example MAP@3:", mapk(actual, predictions))

In [ ]:
# ======================================================
# PHASE 2 - STEP 6 : CREATE TRAINING DATA
# ======================================================

train_samples = []

for _, row in train_df.iterrows():

    prompt = row["prompt"]
    correct = row["answer"]

    for option in ["A", "B", "C", "D", "E"]:

        text = (
            f"Question: {prompt}\n\n"
            f"Option: {row[option]}"
        )

        train_samples.append({
            "id": row["id"],
            "option": option,
            "text": text,
            "label": int(option == correct)
        })

binary_train = pd.DataFrame(train_samples)

print(binary_train.shape)

display(binary_train.head(10))

print("\nPositive Samples :", binary_train["label"].sum())
print("Negative Samples :", len(binary_train) - binary_train["label"].sum())

In [ ]:
# ======================================================
# PHASE 2 - STEP 7.1 : IMPORTS
# ======================================================

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import GroupKFold

from sklearn.metrics import accuracy_score

In [ ]:
# ======================================================
# PHASE 2 - STEP 7.2 : TF-IDF + LOGISTIC REGRESSION
# (GroupKFold - No Data Leakage)
# ======================================================

X = binary_train["text"]
y = binary_train["label"]
groups = binary_train["id"]

gkf = GroupKFold(n_splits=5)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):

    print("=" * 60)
    print(f"Fold {fold}")
    print("=" * 60)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]

    # -----------------------------
    # TF-IDF
    # -----------------------------
    tfidf = TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        stop_words="english"
    )

    X_train_vec = tfidf.fit_transform(X_train)
    X_valid_vec = tfidf.transform(X_valid)

    # -----------------------------
    # Logistic Regression
    # -----------------------------
    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train_vec, y_train)

    # Probability that option is correct
    probs = model.predict_proba(X_valid_vec)[:, 1]

    # -----------------------------
    # Build validation dataframe
    # -----------------------------
    fold_df = binary_train.iloc[valid_idx].copy().reset_index(drop=True)
    fold_df["score"] = probs

    predictions = []
    actuals = []

    # Every question should have exactly 5 options
    for qid, grp in fold_df.groupby("id"):

        grp = grp.sort_values("score", ascending=False)

        # Top-3 predicted options
        pred = grp["option"].head(3).tolist()

        # Correct option
        correct = grp.loc[grp["label"] == 1, "option"]

        if len(correct) != 1:
            print(f"Skipping Question {qid} (unexpected number of correct options)")
            continue

        predictions.append(pred)
        actuals.append(correct.iloc[0])

    score = mapk(actuals, predictions)

    fold_scores.append(score)

    print(f"MAP@3 : {score:.5f}")

print("\n" + "=" * 60)
print("Cross Validation Results")
print("=" * 60)

for i, score in enumerate(fold_scores, start=1):
    print(f"Fold {i}: {score:.5f}")

print("-" * 60)
print(f"Mean MAP@3 : {np.mean(fold_scores):.5f}")

In [ ]:
# ======================================================
# PHASE 2 - STEP 8 : DUPLICATE PROMPT ANALYSIS
# ======================================================

prompt_counts = train_df["prompt"].value_counts()

duplicate_prompts = prompt_counts[prompt_counts > 1]

print(f"Total unique prompts: {train_df['prompt'].nunique()}")
print(f"Duplicate prompts: {len(duplicate_prompts)}")
print(f"Questions involved: {duplicate_prompts.sum()}")

print("\nTop 10 most repeated prompts:\n")
display(duplicate_prompts.head(10))